In [7]:
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
import os
from sqlalchemy import create_engine, text

In [8]:
load_dotenv()

True

In [9]:
db_host = os.getenv("DB_HOST")
db_name = os.getenv("DB_NAME")
db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")
db_port = os.getenv("DB_PORT")

# Use URL.create so special chars (like @) in password are handled correctly
from sqlalchemy.engine import URL

db_url = URL.create(
    drivername="postgresql+psycopg2",
    username=db_user,
    password=db_password,
    host=db_host,
    port=int(db_port) if db_port else None,
    database=db_name,
 )

engine = create_engine(db_url)

In [10]:
csv_path = Path("data/clean/dim_company.csv")
df = pd.read_csv(csv_path)

target_cols = [
    "symbol",
    "company_name",
    "sector_id",
    "sub_sector",
    "company_logo",
    "website",
    "nse_url",
    "bse_url",
    "face_value",
    "book_value",
    "about_company",
]

# Keep only expected columns and convert NaN -> None so PostgreSQL gets NULL
df = df[target_cols].where(pd.notna(df[target_cols]), None)
records = df.to_dict(orient="records")

upsert_sql = text("""
    INSERT INTO dim_company (
        symbol, company_name, sector_id, sub_sector, company_logo,
        website, nse_url, bse_url, face_value, book_value, about_company
    )
    VALUES (
        :symbol, :company_name, :sector_id, :sub_sector, :company_logo,
        :website, :nse_url, :bse_url, :face_value, :book_value, :about_company
    )
    ON CONFLICT (symbol) DO UPDATE SET
        company_name = EXCLUDED.company_name,
        sector_id = EXCLUDED.sector_id,
        sub_sector = EXCLUDED.sub_sector,
        company_logo = EXCLUDED.company_logo,
        website = EXCLUDED.website,
        nse_url = EXCLUDED.nse_url,
        bse_url = EXCLUDED.bse_url,
        face_value = EXCLUDED.face_value,
        book_value = EXCLUDED.book_value,
        about_company = EXCLUDED.about_company
""")

with engine.begin() as conn:
    before_count = conn.execute(text("SELECT COUNT(*) FROM dim_company")).scalar_one()
    print(f"Rows in dim_company before load: {before_count}")

    conn.execute(upsert_sql, records)

    after_count = conn.execute(text("SELECT COUNT(*) FROM dim_company")).scalar_one()
    print(f"Rows in dim_company after load:  {after_count}")
    print(f"CSV rows processed: {len(records)}")

Rows in dim_company before load: 100
Rows in dim_company after load:  101
CSV rows processed: 101


In [11]:
csv_path = Path("data/clean/dim_health_label.csv")
df = pd.read_csv(csv_path)

target_cols = ["label_id", "label_name", "min_score", "max_score", "color_hex"]

df = df[target_cols].where(pd.notna(df[target_cols]), None)
records = df.to_dict(orient="records")

upsert_sql = text("""
    INSERT INTO dim_health_label (
        label_id, label_name, min_score, max_score, color_hex
    )
    VALUES (
        :label_id, :label_name, :min_score, :max_score, :color_hex
    )
    ON CONFLICT (label_id) DO UPDATE SET
        label_name = EXCLUDED.label_name,
        min_score = EXCLUDED.min_score,
        max_score = EXCLUDED.max_score,
        color_hex = EXCLUDED.color_hex
""")

with engine.begin() as conn:
    before_count = conn.execute(text("SELECT COUNT(*) FROM dim_health_label")).scalar_one()
    print(f"Rows in dim_health_label before load: {before_count}")

    conn.execute(upsert_sql, records)

    after_count = conn.execute(text("SELECT COUNT(*) FROM dim_health_label")).scalar_one()
    print(f"Rows in dim_health_label after load:  {after_count}")
    print(f"CSV rows processed: {len(records)}")

Rows in dim_health_label before load: 5
Rows in dim_health_label after load:  5
CSV rows processed: 5


In [12]:
csv_path = Path("data/clean/dim_sector.csv")
df = pd.read_csv(csv_path)

target_cols = ["sector_id", "sector_name", "sector_code", "description"]

df = df[target_cols].where(pd.notna(df[target_cols]), None)
records = df.to_dict(orient="records")

upsert_sql = text("""
    INSERT INTO dim_sector (
        sector_id, sector_name, sector_code, description              
    )
    VALUES (
        :sector_id, :sector_name, :sector_code, :description
    )
    ON CONFLICT (sector_id) DO UPDATE SET
        sector_name = EXCLUDED.sector_name,
        sector_code = EXCLUDED.sector_code,
        description = EXCLUDED.description
""")

with engine.begin() as conn:
    before_count = conn.execute(text("SELECT COUNT(*) FROM dim_sector")).scalar_one()
    print(f"Rows in dim_sector before load: {before_count}")

    conn.execute(upsert_sql, records)

    after_count = conn.execute(text("SELECT COUNT(*) FROM dim_sector")).scalar_one()
    print(f"Rows in dim_sector after load:  {after_count}")
    print(f"CSV rows processed: {len(records)}")

Rows in dim_sector before load: 19
Rows in dim_sector after load:  19
CSV rows processed: 19


In [13]:
csv_path = Path("data/clean/dim_year.csv")
df = pd.read_csv(csv_path)

target_cols = ["year_id", "year_label", "fiscal_year", "quarter", "is_ttm", "is_half_year", "sort_order"]

# Convert to object first, then NaN -> None so PostgreSQL receives NULL correctly
df = df[target_cols].astype(object).where(pd.notna(df[target_cols]), None)
records = df.to_dict(orient="records")

upsert_sql = text("""
    INSERT INTO dim_year (
        year_id, year_label, fiscal_year, quarter, is_ttm, is_half_year, sort_order
    )
    VALUES (
        :year_id, :year_label, :fiscal_year, :quarter, :is_ttm, :is_half_year, :sort_order
    )
    ON CONFLICT (year_id) DO UPDATE SET
        year_label = EXCLUDED.year_label,
        fiscal_year = EXCLUDED.fiscal_year,
        quarter = EXCLUDED.quarter,
        is_ttm = EXCLUDED.is_ttm,
        is_half_year = EXCLUDED.is_half_year,
        sort_order = EXCLUDED.sort_order
""")

with engine.begin() as conn:
    before_count = conn.execute(text("SELECT COUNT(*) FROM dim_year")).scalar_one()
    print(f"Rows in dim_year before load: {before_count}")

    conn.execute(upsert_sql, records)

    after_count = conn.execute(text("SELECT COUNT(*) FROM dim_year")).scalar_one()
    print(f"Rows in dim_year after load:  {after_count}")
    print(f"CSV rows processed: {len(records)}")

Rows in dim_year before load: 55
Rows in dim_year after load:  55
CSV rows processed: 55


In [14]:
csv_path = Path("data/clean/fact_analysis.csv")
df = pd.read_csv(csv_path)

target_cols = [
    "symbol",
    "period_label",
    "compounded_sales_growth_pct",
    "compounded_profit_growth_pct",
    "stock_price_cagr_pct",
    "roe_pct",
]

# Convert to object first, then NaN -> None so PostgreSQL receives NULL
df = df[target_cols].astype(object).where(pd.notna(df[target_cols]), None)

upsert_sql = text("""
    INSERT INTO fact_analysis (
        symbol, period_label, compounded_sales_growth_pct, compounded_profit_growth_pct, stock_price_cagr_pct, roe_pct
    )
    VALUES (
        :symbol, :period_label, :compounded_sales_growth_pct, :compounded_profit_growth_pct, :stock_price_cagr_pct, :roe_pct
    )
    ON CONFLICT (symbol, period_label) DO UPDATE SET
        compounded_sales_growth_pct = EXCLUDED.compounded_sales_growth_pct,
        compounded_profit_growth_pct = EXCLUDED.compounded_profit_growth_pct,
        stock_price_cagr_pct = EXCLUDED.stock_price_cagr_pct,
        roe_pct = EXCLUDED.roe_pct
""")

with engine.begin() as conn:
    # Required so ON CONFLICT(symbol, period_label) can work
    conn.execute(text("""
        CREATE UNIQUE INDEX IF NOT EXISTS ux_fact_analysis_symbol_period
        ON fact_analysis (symbol, period_label)
    """))

    before_count = conn.execute(text("SELECT COUNT(*) FROM fact_analysis")).scalar_one()
    print(f"Rows in fact_analysis before load: {before_count}")

    # Keep only records whose symbol exists in dim_company to avoid FK errors
    valid_symbols = {
        row[0] for row in conn.execute(text("SELECT symbol FROM dim_company"))
    }

    valid_df = df[df["symbol"].isin(valid_symbols)].copy()
    skipped_symbols = sorted(set(df["symbol"]) - valid_symbols)
    records = valid_df.to_dict(orient="records")

    if skipped_symbols:
        print(f"Skipped symbols (not found in dim_company): {', '.join(skipped_symbols)}")

    if records:
        conn.execute(upsert_sql, records)

    after_count = conn.execute(text("SELECT COUNT(*) FROM fact_analysis")).scalar_one()
    print(f"Rows in fact_analysis after load:  {after_count}")
    print(f"CSV rows processed: {len(df)}")
    print(f"Rows loaded/upserted: {len(records)}")

Rows in fact_analysis before load: 20
Rows in fact_analysis after load:  20
CSV rows processed: 20
Rows loaded/upserted: 20


In [15]:
csv_path = Path("data/clean/fact_balance_sheet.csv")
df = pd.read_csv(csv_path)

target_cols = [
    "symbol",
    "year_id",
    "equity_capital",
    "reserves",
    "borrowings",
    "other_liabilities",
    "total_liabilities",
    "fixed_assets",
    "cwip",
    "investments",
    "other_asset",
    "total_assets",
    "debt_to_equity",
    "equity_ratio",
    "shares_outstanding",
    "book_value_per_share",
]

# Convert to object first, then NaN -> None so PostgreSQL receives NULL
df = df[target_cols].astype(object).where(pd.notna(df[target_cols]), None)
records = df.to_dict(orient="records")

upsert_sql = text("""
    INSERT INTO fact_balance_sheet (
        symbol, year_id, equity_capital, reserves, borrowings, other_liabilities,
        total_liabilities, fixed_assets, cwip, investments, other_asset, total_assets,
        debt_to_equity, equity_ratio, shares_outstanding, book_value_per_share
    )
    VALUES (
        :symbol, :year_id, :equity_capital, :reserves, :borrowings, :other_liabilities,
        :total_liabilities, :fixed_assets, :cwip, :investments, :other_asset, :total_assets,
        :debt_to_equity, :equity_ratio, :shares_outstanding, :book_value_per_share
    )
    ON CONFLICT (symbol, year_id) DO UPDATE SET
        equity_capital = EXCLUDED.equity_capital,
        reserves = EXCLUDED.reserves,
        borrowings = EXCLUDED.borrowings,
        other_liabilities = EXCLUDED.other_liabilities,
        total_liabilities = EXCLUDED.total_liabilities,
        fixed_assets = EXCLUDED.fixed_assets,
        cwip = EXCLUDED.cwip,
        investments = EXCLUDED.investments,
        other_asset = EXCLUDED.other_asset,
        total_assets = EXCLUDED.total_assets,
        debt_to_equity = EXCLUDED.debt_to_equity,
        equity_ratio = EXCLUDED.equity_ratio,
        shares_outstanding = EXCLUDED.shares_outstanding,
        book_value_per_share = EXCLUDED.book_value_per_share
""")

with engine.begin() as conn:
    # Required so ON CONFLICT(symbol, year_id) can work
    conn.execute(text("""
        CREATE UNIQUE INDEX IF NOT EXISTS ux_fact_balance_sheet_symbol_year
        ON fact_balance_sheet (symbol, year_id)
    """))

    before_count = conn.execute(text("SELECT COUNT(*) FROM fact_balance_sheet")).scalar_one()
    print(f"Rows in fact_balance_sheet before load: {before_count}")

    conn.execute(upsert_sql, records)

    after_count = conn.execute(text("SELECT COUNT(*) FROM fact_balance_sheet")).scalar_one()
    print(f"Rows in fact_balance_sheet after load:  {after_count}")
    print(f"CSV rows processed: {len(records)}")

Rows in fact_balance_sheet before load: 1225
Rows in fact_balance_sheet after load:  1225
CSV rows processed: 1312


In [16]:
csv_path = Path("data/clean/fact_cash_flow.csv")
df = pd.read_csv(csv_path)

target_cols = [
    "symbol",
    "year_id",
    "operating_activity",
    "investing_activity",
    "financing_activity",
    "net_cash_flow",
    "free_cash_flow",
    "cash_conversion_ratio",
]

# Convert to object first, then NaN -> None so PostgreSQL receives NULL
df = df[target_cols].astype(object).where(pd.notna(df[target_cols]), None)
records = df.to_dict(orient="records")

upsert_sql = text("""
    INSERT INTO fact_cash_flow (
        symbol, year_id, operating_activity, investing_activity, financing_activity,
        net_cash_flow, free_cash_flow, cash_conversion_ratio              
    )
    VALUES (
        :symbol, :year_id, :operating_activity, :investing_activity, :financing_activity,
        :net_cash_flow, :free_cash_flow, :cash_conversion_ratio
    )
    ON CONFLICT (symbol, year_id) DO UPDATE SET
        operating_activity = EXCLUDED.operating_activity,
        investing_activity = EXCLUDED.investing_activity,
        financing_activity = EXCLUDED.financing_activity,
        net_cash_flow = EXCLUDED.net_cash_flow,
        free_cash_flow = EXCLUDED.free_cash_flow,
        cash_conversion_ratio = EXCLUDED.cash_conversion_ratio
""")

with engine.begin() as conn:
    # Required so ON CONFLICT(symbol, year_id) can work
    conn.execute(text("""
        CREATE UNIQUE INDEX IF NOT EXISTS ux_fact_cash_flow_symbol_year
        ON fact_cash_flow (symbol, year_id)
    """))

    before_count = conn.execute(text("SELECT COUNT(*) FROM fact_cash_flow")).scalar_one()
    print(f"Rows in fact_cash_flow before load: {before_count}")

    conn.execute(upsert_sql, records)

    after_count = conn.execute(text("SELECT COUNT(*) FROM fact_cash_flow")).scalar_one()
    print(f"Rows in fact_cash_flow after load:  {after_count}")
    print(f"CSV rows processed: {len(records)}")

Rows in fact_cash_flow before load: 0
Rows in fact_cash_flow after load:  1152
CSV rows processed: 1271


In [17]:
csv_path = Path("data/clean/fact_ml_scores.csv")
df = pd.read_csv(csv_path)

target_cols = [
    "symbol",
    "computed_at",
    "overall_score",
    "profitability_score",
    "growth_score",
    "leverage_score",
    "cashflow_score",
    "dividend_score",
    "trend_score",
    "health_label",
]

# Convert computed_at safely and keep NULLs as None for PostgreSQL
if "computed_at" in df.columns:
    df["computed_at"] = pd.to_datetime(df["computed_at"], errors="coerce")

df = df[target_cols].astype(object).where(pd.notna(df[target_cols]), None)
records = df.to_dict(orient="records")

upsert_sql = text("""
    INSERT INTO fact_ml_scores (
        symbol, computed_at, overall_score, profitability_score, growth_score,
        leverage_score, cashflow_score, dividend_score, trend_score, health_label
    )
    VALUES (
        :symbol, :computed_at, :overall_score, :profitability_score, :growth_score,
        :leverage_score, :cashflow_score, :dividend_score, :trend_score, :health_label
    )
    ON CONFLICT (symbol, computed_at) DO UPDATE SET
        overall_score = EXCLUDED.overall_score,
        profitability_score = EXCLUDED.profitability_score,
        growth_score = EXCLUDED.growth_score,
        leverage_score = EXCLUDED.leverage_score,
        cashflow_score = EXCLUDED.cashflow_score,
        dividend_score = EXCLUDED.dividend_score,
        trend_score = EXCLUDED.trend_score,
        health_label = EXCLUDED.health_label
""")

with engine.begin() as conn:
    # Required so ON CONFLICT(symbol, computed_at) can work
    conn.execute(text("""
        CREATE UNIQUE INDEX IF NOT EXISTS ux_fact_ml_scores_symbol_computed_at
        ON fact_ml_scores (symbol, computed_at)
    """))

    before_count = conn.execute(text("SELECT COUNT(*) FROM fact_ml_scores")).scalar_one()
    print(f"Rows in fact_ml_scores before load: {before_count}")

    if records:
        conn.execute(upsert_sql, records)

    after_count = conn.execute(text("SELECT COUNT(*) FROM fact_ml_scores")).scalar_one()
    print(f"Rows in fact_ml_scores after load:  {after_count}")
    print(f"CSV rows processed: {len(records)}")

Rows in fact_ml_scores before load: 0
Rows in fact_ml_scores after load:  0
CSV rows processed: 0


In [18]:
csv_path = Path("data/clean/fact_profit_loss.csv")
df = pd.read_csv(csv_path)

target_cols = [
    "symbol",
    "year_id",
    "sales",
    "expenses",
    "operating_profit",
    "opm_pct",
    "other_income",
    "interest",
    "depreciation",
    "profit_before_tax",
    "tax_pct",
    "net_profit",
    "eps",
    "dividend_payout_pct",
    "net_profit_margin_pct",
    "expense_ratio_pct",
    "interest_coverage",
    "asset_turnover",
    "return_on_assets",
]

# Convert to object first, then NaN -> None so PostgreSQL receives NULL
df = df[target_cols].astype(object).where(pd.notna(df[target_cols]), None)
records = df.to_dict(orient="records")

upsert_sql = text("""
    INSERT INTO fact_profit_loss (
        symbol, year_id, sales, expenses, operating_profit, opm_pct, other_income, interest,
        depreciation, profit_before_tax, tax_pct, net_profit, eps, dividend_payout_pct,
        net_profit_margin_pct, expense_ratio_pct, interest_coverage, asset_turnover, return_on_assets
    )
    VALUES (
        :symbol, :year_id, :sales, :expenses, :operating_profit, :opm_pct, :other_income, :interest,
        :depreciation, :profit_before_tax, :tax_pct, :net_profit, :eps, :dividend_payout_pct,
        :net_profit_margin_pct, :expense_ratio_pct, :interest_coverage, :asset_turnover, :return_on_assets
    )
    ON CONFLICT (symbol, year_id) DO UPDATE SET
        sales = EXCLUDED.sales,
        expenses = EXCLUDED.expenses,
        operating_profit = EXCLUDED.operating_profit,
        opm_pct = EXCLUDED.opm_pct,
        other_income = EXCLUDED.other_income,
        interest = EXCLUDED.interest,
        depreciation = EXCLUDED.depreciation,
        profit_before_tax = EXCLUDED.profit_before_tax,
        tax_pct = EXCLUDED.tax_pct,
        net_profit = EXCLUDED.net_profit,
        eps = EXCLUDED.eps,
        dividend_payout_pct = EXCLUDED.dividend_payout_pct,
        net_profit_margin_pct = EXCLUDED.net_profit_margin_pct,
        expense_ratio_pct = EXCLUDED.expense_ratio_pct,
        interest_coverage = EXCLUDED.interest_coverage,
        asset_turnover = EXCLUDED.asset_turnover,
        return_on_assets = EXCLUDED.return_on_assets
""")

with engine.begin() as conn:
    # Required so ON CONFLICT(symbol, year_id) can work
    conn.execute(text("""
        CREATE UNIQUE INDEX IF NOT EXISTS ux_fact_profit_loss_symbol_year
        ON fact_profit_loss (symbol, year_id)
    """))

    before_count = conn.execute(text("SELECT COUNT(*) FROM fact_profit_loss")).scalar_one()
    print(f"Rows in fact_profit_loss before load: {before_count}")

    conn.execute(upsert_sql, records)

    after_count = conn.execute(text("SELECT COUNT(*) FROM fact_profit_loss")).scalar_one()
    print(f"Rows in fact_profit_loss after load:  {after_count}")
    print(f"CSV rows processed: {len(records)}")

Rows in fact_profit_loss before load: 0
Rows in fact_profit_loss after load:  1263
CSV rows processed: 1360


In [19]:
csv_path = Path("data/clean/fact_pros_cons.csv")
df = pd.read_csv(csv_path)

target_cols = [
    "symbol",
    "is_pro",
    "category",
    "text",
    "source",
    "confidence",
    "generated_at",
]

# Parse timestamp safely and convert NaN -> None for PostgreSQL
if "generated_at" in df.columns:
    df["generated_at"] = pd.to_datetime(df["generated_at"], errors="coerce")

df = df[target_cols].astype(object).where(pd.notna(df[target_cols]), None)

upsert_sql = text("""
    INSERT INTO fact_pros_cons (
        symbol, is_pro, category, text, source, confidence, generated_at
    )
    VALUES (
        :symbol, :is_pro, :category, :text, :source, :confidence, :generated_at
    )
    ON CONFLICT (symbol, is_pro, category, text, generated_at) DO UPDATE SET
        source = EXCLUDED.source,
        confidence = EXCLUDED.confidence
""")

with engine.begin() as conn:
    # Required so ON CONFLICT(...) can work
    conn.execute(text("""
        CREATE UNIQUE INDEX IF NOT EXISTS ux_fact_pros_cons_unique_row
        ON fact_pros_cons (symbol, is_pro, category, text, generated_at)
    """))

    before_count = conn.execute(text("SELECT COUNT(*) FROM fact_pros_cons")).scalar_one()
    print(f"Rows in fact_pros_cons before load: {before_count}")

    # Keep only records whose symbol exists in dim_company to avoid FK errors
    valid_symbols = {
        row[0] for row in conn.execute(text("SELECT symbol FROM dim_company"))
    }

    valid_df = df[df["symbol"].isin(valid_symbols)].copy()
    skipped_symbols = sorted(set(df["symbol"]) - valid_symbols)
    records = valid_df.to_dict(orient="records")

    if skipped_symbols:
        print(f"Skipped symbols (not found in dim_company): {', '.join(skipped_symbols)}")

    if records:
        conn.execute(upsert_sql, records)

    after_count = conn.execute(text("SELECT COUNT(*) FROM fact_pros_cons")).scalar_one()
    print(f"Rows in fact_pros_cons after load:  {after_count}")
    print(f"CSV rows processed: {len(df)}")
    print(f"Rows loaded/upserted: {len(records)}")

Rows in fact_pros_cons before load: 0
Rows in fact_pros_cons after load:  26
CSV rows processed: 26
Rows loaded/upserted: 26
